In [1]:
import os
import json
import sys
import argparse
import datetime
import pandas as pd
import numpy as np
import traceback
from etc.db_handler.mongodb_client import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
import _pickle as pickle
import time
import datetime
from etc.redis_connector.redis_connector import InitRedis
from multiprocessing import Process
import psycopg2
import pymongo
from kp_info_loader_core import InitCore
from exchange_plugin.binance_plug import InitBinanceAdaptor

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [4]:
# import os
# import sys
# import pymongo
# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# sys.path.append(upper_dir)
# from loggers.logger import KimpBotLogger

# class InitDBClient:
#     def __init__(self, host, port, logging_dir=None):
#         self.logger = None
#         self.host = host
#         self.port = port
#         if logging_dir is not None:
#             self.logger = KimpBotLogger("InitDBClient", logging_dir).logger

#     def get_conn(self):
#         mongo_client = pymongo.MongoClient(self.host, self.port)
#         return mongo_client

In [5]:
logging_dir = "/home/kp_info_loader/"
with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
    config = json.load(f)
proc_n = 5
# node = config['node']
node = 'kp_info_loader'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]
# db_dict['database'] = 'info_core'

# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets_dict = config['enabled_markets']

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [6]:
db_client = InitDBClient(**db_dict)
binance_adaptor = InitBinanceAdaptor()
mongo_db_client = db_client.get_conn()

2023-10-23 11:43:45,710 - binance_plug - INFO - binance_plug_logger started.


2023-10-23 11:44:16,096 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API


In [14]:
exchange = 'BINANCE'
futures_type = 'USD_M'

mongo_db = mongo_db_client[f"{exchange}_fundingrate"]
collection = mongo_db[futures_type]
# get all the data
data = collection.find({})
# convert to dataframe
df = pd.DataFrame(data)

funding_df = binance_adaptor.get_fundingrate('USD_M')
funding_df['datetime_now'] = datetime.datetime.utcnow()

2023-10-23 11:49:44,907 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API


In [15]:
funding_df.head()

,symbol,funding_rate,funding_time,base_asset,quote_asset,perpetual,datetime_now
0,SUSHIUSDT,0.0001,2023-10-23 08:00:00,SUSHI,USDT,True,2023-10-23 02:49:44.956883
1,BTSUSDT,0.0001,2023-10-23 08:00:00,BTS,USDT,True,2023-10-23 02:49:44.956883
2,INJUSDT,-0.000351,2023-10-23 08:00:00,INJ,USDT,True,2023-10-23 02:49:44.956883
3,BNTUSDT,0.0001,2023-10-23 08:00:00,BNT,USDT,True,2023-10-23 02:49:44.956883
4,STRAXUSDT,-0.000308,2023-10-23 04:00:00,STRAX,USDT,True,2023-10-23 02:49:44.956883


In [16]:
df

,_id,symbol,funding_rate,funding_time,base_asset,quote_asset,perpetual,datetime_now
0,65320203876fe0c386835e1c,SUSHIUSDT,0.000095,2023-10-20 08:00:00,SUSHI,USDT,True,2023-10-20 07:59:41.847
1,65320203876fe0c386835e1d,BTSUSDT,0.000100,2023-10-20 08:00:00,BTS,USDT,True,2023-10-20 07:59:41.847
2,65320203876fe0c386835e1e,INJUSDT,-0.000586,2023-10-20 08:00:00,INJ,USDT,True,2023-10-20 07:59:41.847
3,65320203876fe0c386835e1f,BNTUSDT,0.000100,2023-10-20 08:00:00,BNT,USDT,True,2023-10-20 07:59:41.847
4,65320203876fe0c386835e20,STRAXUSDT,-0.000844,2023-10-20 08:00:00,STRAX,USDT,True,2023-10-20 07:59:41.847
...,...,...,...,...,...,...,...,...
261,65320203876fe0c386835f21,LEVERBUSD,0.000100,2023-10-20 08:00:00,LEVER,BUSD,True,2023-10-20 07:53:40.652
262,65320203876fe0c386835f22,CHZUSDT,0.000074,2023-10-20 08:00:00,CHZ,USDT,True,2023-10-20 06:50:19.943
263,65320203876fe0c386835f23,DUSKUSDT,0.000100,2023-10-20 08:00:00,DUSK,USDT,True,2023-10-20 06:50:19.943
264,65320203876fe0c386835f24,CTSIUSDT,0.000100,2023-10-20 08:00:00,CTSI,USDT,True,2023-10-20 07:53:40.652


In [17]:
funding_df.merge(df, on=['symbol','funding_time'], how='left')

,symbol,funding_rate_x,funding_time,base_asset_x,quote_asset_x,perpetual_x,datetime_now_x,_id,funding_rate_y,base_asset_y,quote_asset_y,perpetual_y,datetime_now_y
0,SUSHIUSDT,0.0001,2023-10-23 08:00:00,SUSHI,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
1,BTSUSDT,0.0001,2023-10-23 08:00:00,BTS,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
2,INJUSDT,-0.000351,2023-10-23 08:00:00,INJ,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
3,BNTUSDT,0.0001,2023-10-23 08:00:00,BNT,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
4,STRAXUSDT,-0.000308,2023-10-23 04:00:00,STRAX,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,LEVERBUSD,0.0001,2023-10-23 08:00:00,LEVER,BUSD,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
264,CHZUSDT,-0.000144,2023-10-23 08:00:00,CHZ,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
265,DUSKUSDT,0.0001,2023-10-23 08:00:00,DUSK,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT
266,CTSIUSDT,0.0001,2023-10-23 08:00:00,CTSI,USDT,True,2023-10-23 02:49:44.956883,NaN,NaN,NaN,NaN,NaN,NaT


In [6]:
binance_adaptor = InitBinanceAdaptor(info_dict={})

2023-10-16 19:31:22,827 - binance_plug - INFO - binance_plug_logger started.


In [13]:
# get db name list
db_name_list = [x for x in mongo_db_client.list_database_names() if '__' in x]

In [14]:
db_name_list

['UPBIT_SPOT__BTC-BINANCE_COIN_M__USD',
 'UPBIT_SPOT__BTC-BINANCE_SPOT__BTC',
 'UPBIT_SPOT__BTC-BINANCE_SPOT__BUSD',
 'UPBIT_SPOT__BTC-BINANCE_SPOT__USDT',
 'UPBIT_SPOT__BTC-BINANCE_USD_M__BUSD',
 'UPBIT_SPOT__BTC-BINANCE_USD_M__USDT',
 'UPBIT_SPOT__BTC-UPBIT_SPOT__KRW',
 'UPBIT_SPOT__KRW-BINANCE_COIN_M__USD',
 'UPBIT_SPOT__KRW-BINANCE_SPOT__BTC',
 'UPBIT_SPOT__KRW-BINANCE_SPOT__BUSD',
 'UPBIT_SPOT__KRW-BINANCE_SPOT__USDT',
 'UPBIT_SPOT__KRW-BINANCE_USD_M__BUSD',
 'UPBIT_SPOT__KRW-BINANCE_USD_M__USDT',
 'UPBIT_SPOT__KRW-UPBIT_SPOT__BTC']

In [20]:
df

,_id,symbol,funding_rate,funding_time,datetime_now
0,652d10fc8748265d9c39ce77,LINKUSD_231229,NaN,1970-01-01 09:00:00,2023-10-16 10:31:24.387
1,652d10fc8748265d9c39ce78,FILUSD_PERP,0.0001,2023-10-17 01:00:00,2023-10-16 10:31:24.387
2,652d10fc8748265d9c39ce79,XRPUSD_231229,NaN,1970-01-01 09:00:00,2023-10-16 10:31:24.387
3,652d10fc8748265d9c39ce7a,GMTUSD_PERP,0.0001,2023-10-17 01:00:00,2023-10-16 10:31:24.387
4,652d10fc8748265d9c39ce7b,XTZUSD_PERP,0.0001,2023-10-17 01:00:00,2023-10-16 10:31:24.387
...,...,...,...,...,...
243,652d130e8748265d9c39d27f,ETHUSD_240329,NaN,1970-01-01 09:00:00,2023-10-16 10:40:14.694
244,652d130e8748265d9c39d280,NEARUSD_PERP,0.0001,2023-10-17 01:00:00,2023-10-16 10:40:14.694
245,652d130e8748265d9c39d281,NEARUSD_PERP,0.0001,2023-10-17 01:00:00,2023-10-16 10:40:14.694
246,652d130e8748265d9c39d282,DOTUSD_231229,NaN,1970-01-01 09:00:00,2023-10-16 10:40:14.694


In [21]:
# retreive all

cursor = mongo_db_client["UPBIT_SPOT__KRW-BINANCE_USD_M__USDT"]['ARK_1T'].find({})
temp_df = pd.DataFrame(cursor).drop('_id', axis=1)
temp_df.head()


,base_asset,datetime_now,tp_open,tp_high,tp_low,tp_close,LS_open,LS_high,LS_low,LS_close,...,SL_high,SL_low,SL_close,dollar,record_count,tp,scr,atp24h,converted_tp,closed
0,ARK,2023-10-16 09:02:00,0.954485,1.010654,0.817690,1.010654,1.110021,1.128769,0.954485,0.991924,...,0.973201,0.799007,0.836381,1355.0,174,738.0,-0.806452,7.344468e+10,730.61600,True
1,ARK,2023-10-16 09:03:00,1.010654,1.010654,0.805137,0.885692,0.991924,1.035097,0.885692,0.885692,...,0.879690,0.730535,0.730535,1355.5,263,739.0,-0.672043,7.347407e+10,732.51220,True
2,ARK,2023-10-16 09:04:00,0.885692,0.923043,0.711902,0.885692,0.885692,0.923043,0.811072,0.904364,...,0.767822,0.656044,0.749175,1355.5,203,739.0,-0.672043,7.348174e+10,732.51220,True
3,ARK,2023-10-16 09:05:00,0.885692,1.016542,0.749175,0.979122,0.904364,1.072725,0.904364,0.997828,...,0.917246,0.705836,0.705836,1355.5,206,739.0,-0.672043,7.346836e+10,731.83445,True
4,ARK,2023-10-16 09:06:00,0.979122,0.979122,0.631280,0.767822,0.997828,0.997828,0.786476,0.923043,...,0.786476,0.575436,0.749175,1355.5,189,738.0,-0.806452,7.350327e+10,732.37665,True


In [22]:
total_df = pd.DataFrame()
for db_name in db_name_list:
    mongo_db = mongo_db_client[db_name]
    # get the list of collections
    collection_list = mongo_db.list_collection_names()

    for collection_name in collection_list:
        collection = mongo_db[collection_name]
        df = pd.DataFrame(list(collection.find())).tail(1)
        df['db_name'] = db_name
        df['interval'] = collection_name.split('_')[-1]
        total_df = pd.concat([total_df, df], axis=0)

def check_updated(row):
    if 'T' in row['interval']:
        min_num = int(row['interval'].replace('T',''))
    elif 'H' in row['interval']:
        min_num = int(row['interval'].replace('H','')) * 60
    else:
        raise Exception('interval error')
    
    if ((datetime.datetime.now() - row['datetime_now']).total_seconds() / 60) > (min_num * 2):
        return False
    else:
        return True

# Apply check_updated function for every row
total_df['updated'] = total_df.apply(check_updated, axis=1)

In [ ]:
total_df[total_df['datetime_now']<datetime.datetime.now()-datetime.timedelta(minutes=30)]#['base_asset'].unique()

In [ ]:
collection_list

In [30]:
redis_client = InitRedis()

In [ ]:
redis_client.close_all()

In [74]:
target_market_code = "UPBIT_SPOT/KRW"
# origin_market_code = "BINANCE_USD_M/USDT"
origin_market_code = "UPBIT_SPOT/BTC"

# columns_to_merge = ['base_asset', 'tp', 'scr', 'atp24h', 'converted_tp']

# original_period = '1T'
# resample_period = '5T'
# resample_closed_count = 5
# max_length = 300

# def resample_ohlc_df(ohlc_df, resample_period, closed_count):
#     ohlc_df['count'] = 1
#     ohlc_df = ohlc_df.set_index(['base_asset','datetime_now'])
#     resampled_df = ohlc_df.groupby('base_asset').resample(resample_period, level='datetime_now').agg({
#         'tp_open': 'first',
#         'tp_high': 'max',
#         'tp_low': 'min',
#         'tp_close': 'last',
#         'LS_open': 'first',
#         'LS_high': 'max',
#         'LS_low': 'min',
#         'LS_close': 'last',
#         'SL_open': 'first',
#         'SL_high': 'max',
#         'SL_low': 'min',
#         'SL_close': 'last',
#         'count': 'sum'
#     })
#     resampled_df = resampled_df.reset_index()
#     resampled_df['closed'] = resampled_df['count'].apply(lambda x: True if x==closed_count else False)
#     return resampled_df

# original_ohlc_1T_now = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now'))
# resampled_ohlc_history_df = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_kline'))

# new_resampled_ohlc_history_df = resample_ohlc_df(pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')), resample_period, closed_count=resample_closed_count)
# new_resampled_ohlc_history_df = new_resampled_ohlc_history_df.merge(original_ohlc_1T_now[columns_to_merge], on=['base_asset'], how='inner')
# resampled_ohlc_history_df = pd.concat([resampled_ohlc_history_df, new_resampled_ohlc_history_df], axis=0, ignore_index=True)
# resampled_ohlc_history_df = resampled_ohlc_history_df.drop_duplicates(subset=['base_asset', 'datetime_now'], keep='last').groupby('base_asset').tail(max_length)

df = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now'))

In [ ]:
core = InitCore(logging_dir, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, enabled_markets_dict, db_dict)

In [ ]:
core.get_premium_df("UPBIT_SPOT/KRW", "BINANCE_USD_M/USDT")

In [ ]:
core.check_status()

In [ ]:
core.symbols_to_exclude

In [ ]:
core.remove_symbol_to_exclude('XRP')

In [ ]:
temp = pickle.loads(redis_client.get_data("INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1H_kline"))
temp[temp['base_asset']=='ENJ'].tail()
# temp

In [ ]:
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = 'BINANCE_USD_M/USDT'
appended_premium_df = pd.DataFrame()
for i in range(10):
    time.sleep(0.01)
    # premium_df = self.get_premium_df(target_market_code, origin_market_code)
    premium_df = core.get_premium_df(target_market_code, origin_market_code)
    datetime_now = datetime.datetime.now()
    premium_df['datetime_now'] = datetime_now
    appended_premium_df = pd.concat([appended_premium_df, premium_df], axis=0)
    appended_premium_df.loc[: ,'datetime_now'] = pd.to_datetime(appended_premium_df['datetime_now'])

In [ ]:
def generate_ohlc_df(self, appended_df, freq='1T'):
    df = appended_df.set_index(['base_asset', 'datetime_now'])
    ohlc_df = pd.DataFrame()
    if 'tp_premium' in df.columns:
        df['tp_premium'] = pd.to_numeric(df['tp_premium'], errors='coerce')
        tp_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['tp_premium'].ohlc()
        # add prefix of tp_ to the column names
        tp_ohlc_df.columns = ['tp_' + col for col in tp_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, tp_ohlc_df], axis=1)
    if 'LS_premium' in df.columns:
        df['LS_premium'] = pd.to_numeric(df['LS_premium'], errors='coerce')
        LS_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['LS_premium'].ohlc()
        # add prefix of LS_ to the column names
        LS_ohlc_df.columns = ['LS_' + col for col in LS_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, LS_ohlc_df], axis=1)
    if 'SL_premium' in df.columns:
        df['SL_premium'] = pd.to_numeric(df['SL_premium'], errors='coerce')
        SL_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['SL_premium'].ohlc()
        # add prefix of SL_ to the column names
        SL_ohlc_df.columns = ['SL_' + col for col in SL_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, SL_ohlc_df], axis=1)
    if 'tp_premium' not in df.columns and 'LS_premium' not in df.columns and 'SL_premium' not in df.columns:
        raise ValueError('There is no proper column in the dataframe')
    # TEST
    ohlc_df['dollar'] = df['dollar'].iloc[-1]
    ohlc_df.reset_index(inplace=True)
    # TEST
    ohlc_df['record_count'] = appended_df.groupby('base_asset')['base_asset'].count().values
    return ohlc_df

In [ ]:

def generate_ohlc_df(df, freq='1T'):
    # TEST
    start = time.time()
    df = df.set_index(['base_asset', 'datetime_now'])
    ohlc_df = pd.DataFrame()
    if 'tp_premium' in df.columns:
        df['tp_premium'] = pd.to_numeric(df['tp_premium'], errors='coerce')
        tp_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['tp_premium'].ohlc()
        # add prefix of tp_ to the column names
        tp_ohlc_df.columns = ['tp_' + col for col in tp_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, tp_ohlc_df], axis=1)
    if 'LS_premium' in df.columns:
        df['LS_premium'] = pd.to_numeric(df['LS_premium'], errors='coerce')
        LS_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['LS_premium'].ohlc()
        # add prefix of LS_ to the column names
        LS_ohlc_df.columns = ['LS_' + col for col in LS_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, LS_ohlc_df], axis=1)
    if 'SL_premium' in df.columns:
        df['SL_premium'] = pd.to_numeric(df['SL_premium'], errors='coerce')
        SL_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['SL_premium'].ohlc()
        # add prefix of SL_ to the column names
        SL_ohlc_df.columns = ['SL_' + col for col in SL_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, SL_ohlc_df], axis=1)
    if 'tp_premium' not in df.columns and 'LS_premium' not in df.columns and 'SL_premium' not in df.columns:
        raise ValueError('There is no proper column in the dataframe')
    ohlc_df.reset_index(inplace=True)
    # TEST
    print(f"generate_ohlc_df took {time.time()-start} seconds")
    return ohlc_df

# start = time.time()
# ohlc_df = generate_ohlc_df(appended_premium_df, freq='1T')
# print(f"generating ohlc df: {time.time()-start}")
# ohlc_df

In [ ]:
def draw_ohlc(df, premium_type="tp"):
    df = df.set_index('datetime_now', drop=True)
    selected_columns = [col for col in df.columns if premium_type in col]
    df = df[selected_columns]
    df.columns = [col.replace(premium_type+'_', '') for col in df.columns]
    df.rename(columns={'open':'Open', 'high':'High', 'low':'Low', 'close':'Close'}, inplace=True)
    mpf.plot(df, type='candle', style='charles')

In [ ]:
def ohlc_1T_loader(self, get_premium_df, target_market_code, origin_market_code, loop_downtime_sec=0.01, max_length=300):
    columns_to_merge = ['base_asset', 'tp', 'scr', 'atp24h', 'converted_tp']
    appended_premium_df = pd.DataFrame()
    kline_switch = False
    while True:
        time.sleep(loop_downtime_sec)
        # premium_df = self.get_premium_df(target_market_code, origin_market_code)
        premium_df = get_premium_df(target_market_code, origin_market_code)
        datetime_now = datetime.datetime.now()
        premium_df['datetime_now'] = datetime_now
        appended_premium_df = pd.concat([appended_premium_df, premium_df], axis=0)
        appended_premium_df.loc[: ,'datetime_now'] = pd.to_datetime(appended_premium_df['datetime_now'])
        # print(f"redis saving ohlc_1T_now time: {time.time()-start}")
        if datetime_now.second == 0 and kline_switch == True:
            adjusted_datetime_now = datetime.datetime(datetime_now.year, datetime_now.month, datetime_now.day, datetime_now.hour, datetime_now.minute)
            cut_appended_premium_df = appended_premium_df[appended_premium_df['datetime_now'] < adjusted_datetime_now]
            ohlc_df = self.generate_ohlc_df(cut_appended_premium_df)
            ohlc_df = ohlc_df.merge(premium_df[['base_asset','tp','scr','atp24h','converted_tp']], on=['base_asset'], how='inner') # TEST
            pickled_ohlc_df = pickle.dumps(ohlc_df)
            # Save into redis db for current data
            self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now', pickled_ohlc_df)
            # Publish to redis pubsub
            self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now', pickled_ohlc_df)
            # Append into redis db for historical data
            old_ohlc_1T_kline = self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_kline')
            if old_ohlc_1T_kline is None:
                old_ohlc_1T_kline = pd.DataFrame()
            else:
                old_ohlc_1T_kline = pickle.loads(old_ohlc_1T_kline)
            new_ohlc_1T_kline = pd.concat([old_ohlc_1T_kline, ohlc_df], axis=0).tail(max_length*ohlc_df['base_asset'].nunique())
            new_ohlc_1T_kline['closed'] = True
            pickled_ohlc_df = pickle.dumps(new_ohlc_1T_kline)
            self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_kline', pickled_ohlc_df)
            # Publish to redis pubsub
            self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_kline', pickled_ohlc_df)
            appended_premium_df = appended_premium_df[appended_premium_df['datetime_now'] >= adjusted_datetime_now]
            kline_switch = False
        elif datetime_now.second != 0:
            # Save into redis db for current data
            ohlc_df = self.generate_ohlc_df(appended_premium_df)
            ohlc_df = ohlc_df.merge(premium_df[['base_asset','tp','scr','atp24h','converted_tp']], on=['base_asset'], how='inner')
            pickled_ohlc_df = pickle.dumps(ohlc_df)
            self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now', pickled_ohlc_df)
            # Publish to redis pubsub
            self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now', pickled_ohlc_df)
            kline_switch = True
        else:
            # Save into redis db for current data
            ohlc_df = self.generate_ohlc_df(appended_premium_df)
            ohlc_df = ohlc_df.merge(premium_df[['base_asset','tp','scr','atp24h','converted_tp']], on=['base_asset'], how='inner')
            pickled_ohlc_df = pickle.dumps(ohlc_df)
            self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now', pickled_ohlc_df)
            # Publish to redis pubsub
            self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now', pickled_ohlc_df)

In [ ]:
test_list = ['BINANCE_USD_M/USDT','UPBIT_SPOT/KRW','UPBIT_SPOT/BTC','BINANCE_USD_M/BUSD','BINANCE_SPOT/USDT','BINANCE_SPOT/BTC']
test_list = ['BINANCE_USD_M/USDT','UPBIT_SPOT/KRW','UPBIT_SPOT/BTC']
# get all possible combinations between test_list1 elements and test_list2 elements
possible_combinantion_list = []
for i in test_list:
    for j in test_list:
        if i != j:
            possible_combinantion_list.append(f"{i}:{j}")
possible_combinantion_list

In [ ]:
temp = pickle.loads(redis_client.get_data('INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_now'))
temp[temp['base_asset']=='BTC']

In [ ]:
temp = pickle.loads(redis_client.get_data('INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_kline'))
temp.groupby('base_asset').tail(1)

In [ ]:
def resample_ohlc_df(ohlc_df, resample_period, closed_count):
    ohlc_df['count'] = 1
    ohlc_df = ohlc_df.set_index(['base_asset','datetime_now'])
    resampled_df = ohlc_df.groupby('base_asset').resample(resample_period, level='datetime_now').agg({
        'tp_open': 'first',
        'tp_high': 'max',
        'tp_low': 'min',
        'tp_close': 'last',
        'LS_open': 'first',
        'LS_high': 'max',
        'LS_low': 'min',
        'LS_close': 'last',
        'SL_open': 'first',
        'SL_high': 'max',
        'SL_low': 'min',
        'SL_close': 'last',
        'count': 'sum'
    })
    resampled_df = resampled_df.reset_index()
    resampled_df['closed'] = resampled_df['count'].apply(lambda x: True if x==closed_count else False)
    return resampled_df

In [ ]:
def ohlc_hour_resample_loader(self, target_market_code, origin_market_code, original_period, resample_period, resample_closed_count, loop_downtime_sec=0.1, max_length=300):
    columns_to_merge = ['base_asset', 'tp', 'scr', 'atp24h', 'converted_tp']
    while True:
        original_ohlc_kline_df = self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')
        if original_ohlc_kline_df is not None:
            break
        time.sleep(5)

    start = time.time()
    original_ohlc_1T_now = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now'))
    resampled_ohlc_history_df = self.resample_ohlc_df(pickle.loads(original_ohlc_kline_df), resample_period, closed_count=resample_closed_count)
    resampled_ohlc_history_df = resampled_ohlc_history_df.merge(original_ohlc_1T_now[columns_to_merge], on=['base_asset'], how='inner')
    pickled_resampled_ohlc_history_df = pickle.dumps(resampled_ohlc_history_df)
    self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_kline', pickled_resampled_ohlc_history_df)
    # Publish to redis pubsub
    self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_kline', pickled_resampled_ohlc_history_df)
    self.kline_logger.info(f"ohlc_hour_resample_loader has started. {target_market_code}:{origin_market_code}_{resample_period}_kline, initial generating and storing resampled_ohlc_history_df(length: {len(resampled_ohlc_history_df)}): {time.time()-start}")

    while True:
        time.sleep(loop_downtime_sec)
        original_ohlc_1T_now = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now'))
        try:
            original_ohlc_history_last_datetime = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')).iloc[-1]['datetime_now']
        except TypeError:
            print(f"original_ohlc_history_last_datetime is None")
            continue
        resampled_ohlc_history_df = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_kline'))
        # resampled_ohlc_history_df = resampled_ohlc_history_df[resampled_ohlc_history_df['closed']==True]
        # if resampled_ohlc_history_df.iloc[-1]['closed'] == True:
        #     resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now']
        # else:
        #     resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now'] - datetime.timedelta(minutes=int(resample_period[:-1]))
        resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now']
        datetime_now = datetime.datetime.now()
        # print(f"(original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1]): {(original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1])}")
        # print(f"datetime_now - resampled_ohlc_history_last_datetime > datetime.timedelta(minutes=int(resample_period[:-1])*2): {datetime_now - resampled_ohlc_history_last_datetime > datetime.timedelta(minutes=int(resample_period[:-1])*2)}")
        # print(f"datetime_now: {datetime_now}")
        # print(f"resampled_ohlc_history_last_datetime: {resampled_ohlc_history_last_datetime}")
        # print(f"(datetime_now - resampled_ohlc_history_last_datetime): {datetime_now - resampled_ohlc_history_last_datetime}")
        # if ((original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1]) == 0 and
        #     (datetime_now - resampled_ohlc_history_last_datetime) > datetime.timedelta(minutes=int(resample_period[:-1])*2)):
        if ((original_ohlc_history_last_datetime.hour + int(original_period[:-1])) % int(resample_period[:-1]) == 0 and                
            (datetime_now - resampled_ohlc_history_last_datetime) > datetime.timedelta(hours=int(resample_period[:-1])*2) or
            sorted(resampled_ohlc_history_df['base_asset'].unique()) != sorted(original_ohlc_1T_now['base_asset'].unique())):
            # concatenate fetched resampled_ohlc_history_df and newly generated resampled_ohlc_history_df
            start = time.time()
            new_resampled_ohlc_history_df = self.resample_ohlc_df(pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')), resample_period, closed_count=resample_closed_count)
            new_resampled_ohlc_history_df = new_resampled_ohlc_history_df.merge(original_ohlc_1T_now[columns_to_merge], on=['base_asset'], how='inner')
            resampled_ohlc_history_df = pd.concat([resampled_ohlc_history_df, new_resampled_ohlc_history_df], axis=0, ignore_index=True)
            resampled_ohlc_history_df = resampled_ohlc_history_df.drop_duplicates(subset=['base_asset', 'datetime_now'], keep='last').groupby('base_asset').tail(max_length)
            print(f"resampling ohlc_df(length: {len(resampled_ohlc_history_df)}): {time.time()-start}")
            # Save into the Redis DB
            # start = time.time()
            pickled_resampled_ohlc_history_df = pickle.dumps(resampled_ohlc_history_df)
            self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_kline', pickled_resampled_ohlc_history_df)
            # Publish to redis pubsub
            self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_kline', pickled_resampled_ohlc_history_df)
            # self.kline_logger.info(f"redis saving {target_market_code}:{origin_market_code}_{resample_period}_kline time: {time.time()-start}")
            if sorted(resampled_ohlc_history_df['base_asset'].unique()) != sorted(original_ohlc_1T_now['base_asset'].unique()):
                self.kline_logger.info(f"base_asset is not matched. difference: {[x for x in resampled_ohlc_history_df['base_asset'].unique() if x not in original_ohlc_1T_now['base_asset'].unique()]}")
                time.sleep(5)
        else:
            # generate resampled_ohlc_now_df
            resampled_ohlc_last_df = resampled_ohlc_history_df.groupby('base_asset').tail(1).reset_index(drop=True).copy()
            try:
                # Compare it to thr original_ohlc_1T_now
                resampled_ohlc_last_df.loc[:, 'tp_high'] = np.where(resampled_ohlc_last_df['tp_high']>original_ohlc_1T_now['tp_high'], resampled_ohlc_last_df['tp_high'], original_ohlc_1T_now['tp_high'])
                resampled_ohlc_last_df.loc[:, 'tp_low'] = np.where(resampled_ohlc_last_df['tp_low']<original_ohlc_1T_now['tp_low'], resampled_ohlc_last_df['tp_low'], original_ohlc_1T_now['tp_low'])
                resampled_ohlc_last_df.loc[:, 'tp_close'] = original_ohlc_1T_now['tp_close']
                resampled_ohlc_last_df.loc[:, 'LS_high'] = np.where(resampled_ohlc_last_df['LS_high']>original_ohlc_1T_now['LS_high'], resampled_ohlc_last_df['LS_high'], original_ohlc_1T_now['LS_high'])
                resampled_ohlc_last_df.loc[:, 'LS_low'] = np.where(resampled_ohlc_last_df['LS_low']<original_ohlc_1T_now['LS_low'], resampled_ohlc_last_df['LS_low'], original_ohlc_1T_now['LS_low'])
                resampled_ohlc_last_df.loc[:, 'LS_close'] = original_ohlc_1T_now['LS_close']
                resampled_ohlc_last_df.loc[:, 'SL_high'] = np.where(resampled_ohlc_last_df['SL_high']>original_ohlc_1T_now['SL_high'], resampled_ohlc_last_df['SL_high'], original_ohlc_1T_now['SL_high'])
                resampled_ohlc_last_df.loc[:, 'SL_low'] = np.where(resampled_ohlc_last_df['SL_low']<original_ohlc_1T_now['SL_low'], resampled_ohlc_last_df['SL_low'], original_ohlc_1T_now['SL_low'])
                resampled_ohlc_last_df.loc[:, 'SL_close'] = original_ohlc_1T_now['SL_close']
                resampled_ohlc_last_df[columns_to_merge] = original_ohlc_1T_now[columns_to_merge]
                resampled_ohlc_last_df.loc[:, 'datetime_now'] = resampled_ohlc_last_df['datetime_now'] + datetime.timedelta(hours=int(resample_period[:-1]))
                # # Save into the Redis DB
                # self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_now', pickle.dumps(resampled_ohlc_last_df))
                # PUBSUB
                self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_{resample_period}_now', pickle.dumps(resampled_ohlc_last_df))
            except Exception as e:
                self.kline_logger.error(f"Exception: {e}, \nError in ohlc_hour_resample_loader: {e}, resampled_ohlc_last_df: {resampled_ohlc_last_df}, original_ohlc_1T_now: {original_ohlc_1T_now}")
                time.sleep(3)

def ohlc_min_resample_loader(self, target_market_code, origin_market_code, original_period, resample_period, resample_closed_count, loop_downtime_sec=0.1, max_length=300):
    columns_to_merge = ['base_asset', 'tp', 'scr', 'atp24h', 'converted_tp']
    if resample_period == "60T":
        converted_resample_period = "1H"
    else:
        converted_resample_period = resample_period

    while True:
        original_ohlc_kline_df = self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')
        if original_ohlc_kline_df is not None:
            break
        time.sleep(5)

    start = time.time()
    original_ohlc_1T_now = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now'))
    resampled_ohlc_history_df = self.resample_ohlc_df(pickle.loads(original_ohlc_kline_df), resample_period, closed_count=resample_closed_count)
    resampled_ohlc_history_df = resampled_ohlc_history_df.merge(original_ohlc_1T_now[columns_to_merge], on=['base_asset'], how='inner')
    pickled_resampled_ohlc_history_df = pickle.dumps(resampled_ohlc_history_df)
    self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline', pickled_resampled_ohlc_history_df)
    # Publish to redis pubsub
    self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline', pickled_resampled_ohlc_history_df)
    self.kline_logger.info(f"ohlc_min_resample_loader has started. {target_market_code}:{origin_market_code}_{converted_resample_period}_kline, initial generating and storing resampled_ohlc_history_df(length: {len(resampled_ohlc_history_df)}): {time.time()-start}")

    while True:
        time.sleep(loop_downtime_sec)
        original_ohlc_1T_now = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_1T_now'))
        try:
            original_ohlc_history_last_datetime = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')).iloc[-1]['datetime_now']
        except TypeError:
            print(f"original_ohlc_history_last_datetime is None")
            continue
        resampled_ohlc_history_df = pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline'))
        # resampled_ohlc_history_df = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline'))
        # resampled_ohlc_history_df = resampled_ohlc_history_df[resampled_ohlc_history_df['closed']==True]
        # if resampled_ohlc_history_df.iloc[-1]['closed'] == True:
        #     resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now']
        # else:
        #     resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now'] - datetime.timedelta(minutes=int(resample_period[:-1]))
        resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now']
        datetime_now = datetime.datetime.now()
        # print(f"(original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1]): {(original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1])}")
        # print(f"datetime_now - resampled_ohlc_history_last_datetime > datetime.timedelta(minutes=int(resample_period[:-1])*2): {datetime_now - resampled_ohlc_history_last_datetime > datetime.timedelta(minutes=int(resample_period[:-1])*2)}")
        # print(f"datetime_now: {datetime_now}")
        # print(f"resampled_ohlc_history_last_datetime: {resampled_ohlc_history_last_datetime}")
        # print(f"(datetime_now - resampled_ohlc_history_last_datetime): {datetime_now - resampled_ohlc_history_last_datetime}")
        if ((original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1]) == 0 and
            (datetime_now - resampled_ohlc_history_last_datetime) > datetime.timedelta(minutes=int(resample_period[:-1])*2) or
            sorted(resampled_ohlc_history_df['base_asset'].unique()) != sorted(original_ohlc_1T_now['base_asset'].unique())):
            # concatenate fetched resampled_ohlc_history_df and newly generated resampled_ohlc_history_df
            start = time.time()
            new_resampled_ohlc_history_df = self.resample_ohlc_df(pickle.loads(self.redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline')), resample_period, closed_count=resample_closed_count)
            new_resampled_ohlc_history_df = new_resampled_ohlc_history_df.merge(original_ohlc_1T_now[columns_to_merge], on=['base_asset'], how='inner')
            resampled_ohlc_history_df = pd.concat([resampled_ohlc_history_df, new_resampled_ohlc_history_df], axis=0, ignore_index=True)
            resampled_ohlc_history_df = resampled_ohlc_history_df.drop_duplicates(subset=['base_asset', 'datetime_now'], keep='last').groupby('base_asset').tail(max_length)
            print(f"resampling ohlc_df(length: {len(resampled_ohlc_history_df)}): {time.time()-start}")
            # Save into the Redis DB
            start = time.time()
            pickled_resampled_ohlc_history_df = pickle.dumps(resampled_ohlc_history_df)
            self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline', pickled_resampled_ohlc_history_df)
            self.kline_logger.info(f"redis saving {target_market_code}:{origin_market_code}_{converted_resample_period}_kline time: {time.time()-start}")
            # Publish to redis pubsub
            start = time.time() # TEST
            self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline', pickled_resampled_ohlc_history_df)
            self.kline_logger.info(f"redis publishing {target_market_code}:{origin_market_code}_{converted_resample_period}_kline time: {time.time()-start}") # TEST
            if sorted(resampled_ohlc_history_df['base_asset'].unique()) != sorted(original_ohlc_1T_now['base_asset'].unique()):
                self.kline_logger.info(f"base_asset is not matched. difference: {[x for x in resampled_ohlc_history_df['base_asset'].unique() if x not in original_ohlc_1T_now['base_asset'].unique()]}")
                time.sleep(5)
        else:
            # generate resampled_ohlc_now_df
            resampled_ohlc_last_df = resampled_ohlc_history_df.groupby('base_asset').tail(1).reset_index(drop=True).copy()
            try:
                # Compare it to thr original_ohlc_1T_now
                resampled_ohlc_last_df.loc[:, 'tp_high'] = np.where(resampled_ohlc_last_df['tp_high']>original_ohlc_1T_now['tp_high'], resampled_ohlc_last_df['tp_high'], original_ohlc_1T_now['tp_high'])
                resampled_ohlc_last_df.loc[:, 'tp_low'] = np.where(resampled_ohlc_last_df['tp_low']<original_ohlc_1T_now['tp_low'], resampled_ohlc_last_df['tp_low'], original_ohlc_1T_now['tp_low'])
                resampled_ohlc_last_df.loc[:, 'tp_close'] = original_ohlc_1T_now['tp_close']
                resampled_ohlc_last_df.loc[:, 'LS_high'] = np.where(resampled_ohlc_last_df['LS_high']>original_ohlc_1T_now['LS_high'], resampled_ohlc_last_df['LS_high'], original_ohlc_1T_now['LS_high'])
                resampled_ohlc_last_df.loc[:, 'LS_low'] = np.where(resampled_ohlc_last_df['LS_low']<original_ohlc_1T_now['LS_low'], resampled_ohlc_last_df['LS_low'], original_ohlc_1T_now['LS_low'])
                resampled_ohlc_last_df.loc[:, 'LS_close'] = original_ohlc_1T_now['LS_close']
                resampled_ohlc_last_df.loc[:, 'SL_high'] = np.where(resampled_ohlc_last_df['SL_high']>original_ohlc_1T_now['SL_high'], resampled_ohlc_last_df['SL_high'], original_ohlc_1T_now['SL_high'])
                resampled_ohlc_last_df.loc[:, 'SL_low'] = np.where(resampled_ohlc_last_df['SL_low']<original_ohlc_1T_now['SL_low'], resampled_ohlc_last_df['SL_low'], original_ohlc_1T_now['SL_low'])
                resampled_ohlc_last_df.loc[:, 'SL_close'] = original_ohlc_1T_now['SL_close']
                resampled_ohlc_last_df[columns_to_merge] = original_ohlc_1T_now[columns_to_merge]
                resampled_ohlc_last_df.loc[:, 'datetime_now'] = resampled_ohlc_last_df['datetime_now'] + datetime.timedelta(minutes=int(resample_period[:-1]))
                # # Save into the Redis DB
                # self.redis_client.set_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_now', pickle.dumps(resampled_ohlc_last_df))
                # PUBSUB
                self.redis_client.publish(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_now', pickle.dumps(resampled_ohlc_last_df))
            except Exception as e:
                self.kline_logger.error(f"Exception: {e}, \nError in ohlc_min_resample_loader: {e}, resampled_ohlc_last_df: {resampled_ohlc_last_df}, original_ohlc_1T_now: {original_ohlc_1T_now}")
                time.sleep(3)

In [ ]:
s = redis_client.redis_conn.pubsub()
s.subscribe('INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_now')

In [ ]:
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
original_period = "1T"
converted_resample_period = "5T"
resample_period = "5T"

datetime_now = datetime.datetime.now()
original_ohlc_history_df = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{original_period}_kline'))
original_ohlc_history_last_datetime = original_ohlc_history_df.iloc[-1]['datetime_now']
resampled_ohlc_history_df = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline'))
# resampled_ohlc_history_df = resampled_ohlc_history_df[resampled_ohlc_history_df['closed']==True]
if resampled_ohlc_history_df.iloc[-1]['closed'] == True:
    resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now']
else:
    # resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now'] + datetime.timedelta(minutes=int(resample_period[:-1]))
    resampled_ohlc_history_last_datetime = resampled_ohlc_history_df.iloc[-1]['datetime_now']

In [ ]:
(original_ohlc_history_last_datetime.minute + int(original_period[:-1])) % int(resample_period[:-1]) == 0

In [ ]:
resampled_ohlc_history_df['base_asset'].unique()

In [ ]:
original_ohlc_1T_now['base_asset'].unique()

In [ ]:
original_ohlc_history_df['base_asset'].unique()

In [ ]:
sorted(resampled_ohlc_history_df['base_asset'].unique()) == sorted(original_ohlc_1T_now['base_asset'].unique())

In [ ]:
resampled_ohlc_history_df = pickle.loads(redis_client.get_data(f'INFO_CORE|{target_market_code}:{origin_market_code}_{converted_resample_period}_kline'))
resampled_ohlc_history_df[resampled_ohlc_history_df['base_asset']=='XRP']

In [ ]:
ohlc_loader_proc = Process(target=ohlc_min_resample_loader, args=(core.kline_generator, "UPBIT_SPOT/KRW", "BINANCE_USD_M/USDT", '1T', '5T', 5), daemon=True)
ohlc_loader_proc.start()

In [ ]:
ohlc_loader_proc.terminate()

In [ ]:
ohlc_loader_proc2 = Process(target=ohlc_min_resample_loader, args=(core.kline_generator, "UPBIT_SPOT/KRW", "BINANCE_USD_M/USDT", "5T", "15T", 3), daemon=True)
ohlc_loader_proc2.start()

In [ ]:
ohlc_loader_proc3 = Process(target=ohlc_min_resample_loader, args=("UPBIT_SPOT/KRW", "BINANCE_USD_M/USDT", "15T", "30T", 2), daemon=True)
ohlc_loader_proc3.start()

In [ ]:
ohlc_loader_proc4 = Process(target=ohlc_min_resample_loader, args=("UPBIT_SPOT/KRW", "BINANCE_USD_M/USDT", "30T", "60T", 2), daemon=True)
ohlc_loader_proc4.start()

In [ ]:
ohlc_loader_proc5 = Process(target=ohlc_hour_resample_loader, args=("UPBIT_SPOT/KRW", "BINANCE_USD_M/USDT", "1H", "4H", 4), daemon=True)
ohlc_loader_proc5.start()

In [ ]:
ohlc_loader_proc.terminate()
ohlc_loader_proc2.terminate()
ohlc_loader_proc3.terminate()
ohlc_loader_proc4.terminate()
ohlc_loader_proc5.terminate()

In [ ]:
period = "1H"

temp = pickle.loads(redis_client.get_data(f'INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_{period}_now'))
temp[temp['base_asset'].isin(['XRP','KAVA','HIFI','SEI'])]#[['base_asset','datetime_now','LS_close','SL_close']]

In [ ]:
temp = pickle.loads(redis_client.get_data(f'INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_{period}_kline'))
temp[temp['base_asset'].isin(['HIFI'])].tail(20)

In [ ]:
ohlc_loader_proc.terminate()

In [ ]:
ohlc_loader_proc.is_alive()

In [ ]:
core.check_status(print_result=True)